# Data Transformation — INE Encuesta de Ocupación Hotelera (20/07/2026)

Este notebook parte del fichero creado por:

```text
Data_Cleaning_INE_EOH_20_07_2026.ipynb
```

y actualiza **el mismo clean dataset**, sin crear tablas `fact_`, `dim_`, resúmenes ni otros CSV:

```text
Data/clean_dataset_INE_EOH_20_07_2026.csv
```

## Decisiones aplicadas

- Se conservan todas las filas del clean dataset de Data Cleaning.
- `valor` se valida y normaliza como variable numérica, respetando el formato europeo.
- `periodo` se convierte a una fecha mensual real.
- Se crean variables temporales: año, mes, nombre del mes, trimestre y orden cronológico.
- Se formalizan las variables derivadas utilizadas en el EDA: estancia media, peso nacional/extranjero, correspondencia territorial y temporada de demanda.
- Los porcentajes de la tabla 2069 se ponderan con los totales nacionales de la tabla 2074, sin crear un segundo fichero.
- Las tablas siguen diferenciadas mediante `tabla`, `nivel_geo`, `unidad` y `nivel_jerarquico`.
- El CSV final reemplaza al clean dataset anterior mediante escritura atómica.

> En este contexto, “normalización numérica” significa convertir y validar correctamente los valores numéricos. No se aplica escalado 0–1, porque el dataset mezcla personas, noches y porcentajes.

## 1. Librerías y localización reproducible del proyecto

La carpeta `Equip_34` se localiza automáticamente. El notebook funciona desde `Equip_34`, desde `Scripts` o desde la raíz `ProjecteData`.

No se imprimen rutas personales de macOS o Windows.

In [11]:
from pathlib import Path
import os

import numpy as np
import pandas as pd
from pandas.api.types import is_numeric_dtype
from IPython.display import display

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 60)

NOMBRE_DATASET = "clean_dataset_INE_EOH_20_07_2026.csv"


def encontrar_raiz_proyecto(nombre_carpeta="Equip_34"):
    """Localiza Equip_34 desde el directorio actual o desde uno de sus padres."""
    actual = Path.cwd().resolve()

    for carpeta in [actual, *actual.parents]:
        if carpeta.name == nombre_carpeta:
            return carpeta

        candidata = carpeta / nombre_carpeta
        if (candidata / "Data").is_dir() and (candidata / "Scripts").is_dir():
            return candidata

    raise FileNotFoundError(
        f"No se encontró la carpeta {nombre_carpeta}. "
        "Abre el repositorio ProjecteData o ejecuta el notebook desde Equip_34."
    )


RAIZ_PROYECTO = encontrar_raiz_proyecto()
RUTA_DATASET = RAIZ_PROYECTO / "Data" / NOMBRE_DATASET

if not RUTA_DATASET.exists():
    raise FileNotFoundError(
        f"No se encontró {NOMBRE_DATASET}. "
        "Ejecuta primero Data_Cleaning_INE_EOH_20_07_2026.ipynb."
    )

print(f"Dataset localizado: {NOMBRE_DATASET}")

Dataset localizado: clean_dataset_INE_EOH_20_07_2026.csv


## 2. Carga y validación del clean dataset de entrada

El fichero se carga con el mismo separador, codificación y convención decimal utilizados en Data Cleaning.

In [12]:
COLUMNAS_BASE = [
    "tabla", "nivel_geo", "codigo_ine", "geo", "geo_key", "zona_ccaa",
    "dimension", "categoria", "categoria_key", "metrica", "unidad",
    "nivel_jerarquico", "periodo", "anio", "mes", "fecha", "valor",
    "sin_dato", "provisional", "serie_enlazada",
]

df = pd.read_csv(
    RUTA_DATASET,
    sep=";",
    decimal=",",
    encoding="utf-8-sig",
    dtype={
        "tabla": "string",
        "codigo_ine": "string",
        "periodo": "string",
    },
    # Si el notebook se vuelve a ejecutar sobre el dataset ya transformado,
    # solo se leen las columnas base y las derivadas se recalculan.
    usecols=lambda columna: columna in COLUMNAS_BASE,
    low_memory=False,
)

columnas_ausentes = [col for col in COLUMNAS_BASE if col not in df.columns]
if columnas_ausentes:
    raise ValueError(
        "El clean dataset no contiene todas las columnas esperadas: "
        f"{columnas_ausentes}"
    )

filas_iniciales = len(df)

print(f"Filas cargadas: {filas_iniciales:,}")
print(f"Columnas de entrada: {len(df.columns)}")

Filas cargadas: 1,040,692
Columnas de entrada: 20


## 3. Normalización de valores numéricos y flags

`valor` ya fue tratado en Data Cleaning, pero aquí se vuelve a validar de forma defensiva para que el proceso sea reproducible e idempotente.

Las marcas `.` y `..` ya están representadas mediante `sin_dato`; no se convierten en cero.

In [13]:
def normalizar_numerico(serie):
    """Convierte números estándar o europeos sin alterar los valores ya numéricos."""
    if is_numeric_dtype(serie):
        return pd.to_numeric(serie, errors="coerce")

    conversion_directa = pd.to_numeric(serie, errors="coerce")
    texto = serie.astype("string").str.strip()

    conversion_europea = pd.to_numeric(
        texto.str.replace(".", "", regex=False).str.replace(",", ".", regex=False),
        errors="coerce",
    )

    return conversion_directa.fillna(conversion_europea)


def convertir_booleano(serie):
    """Normaliza flags booleanos leídos desde CSV."""
    if serie.dtype == bool:
        return serie

    mapa = {
        "true": True,
        "false": False,
        "1": True,
        "0": False,
    }

    return (
        serie.astype("string")
        .str.strip()
        .str.lower()
        .map(mapa)
        .fillna(False)
        .astype(bool)
    )


for columna_texto in [
    "tabla", "nivel_geo", "geo", "geo_key", "dimension", "categoria",
    "categoria_key", "metrica", "unidad", "nivel_jerarquico", "periodo",
]:
    df[columna_texto] = df[columna_texto].astype("string").str.strip()

df["valor"] = normalizar_numerico(df["valor"])

for columna_flag in ["sin_dato", "provisional", "serie_enlazada"]:
    df[columna_flag] = convertir_booleano(df[columna_flag])

df["sin_dato"] = df["sin_dato"] | df["valor"].isna()

# Nombre explícito solicitado por el EDA:
# significa dato no publicado / secreto estadístico, sin afirmar cuál de las dos causas aplica.
df["es_dato_no_publicado"] = df["sin_dato"]

print("Tipo final de valor:", df["valor"].dtype)
print("Valores no publicados:", f"{df['es_dato_no_publicado'].sum():,}")

Tipo final de valor: float64
Valores no publicados: 99,588


## 4. Conversión de `periodo` y variables temporales

Transformación principal solicitada por el equipo:

```text
2023M01 → 2023-01-01
```

A partir de `fecha` se generan las variables necesarias para series temporales y Power BI.

In [14]:
periodo_separado = df["periodo"].str.extract(r"^(\d{4})M(\d{2})$")

periodos_invalidos = periodo_separado.isna().any(axis=1)
if periodos_invalidos.any():
    ejemplos = df.loc[periodos_invalidos, "periodo"].drop_duplicates().head().tolist()
    raise ValueError(f"Se encontraron periodos con formato no válido: {ejemplos}")

df["anio"] = periodo_separado[0].astype("int16")
df["mes"] = periodo_separado[1].astype("int8")

df["fecha"] = pd.to_datetime(
    {
        "year": df["anio"].astype(int),
        "month": df["mes"].astype(int),
        "day": 1,
    },
    errors="raise",
)

MESES = {
    1: "Enero", 2: "Febrero", 3: "Marzo", 4: "Abril",
    5: "Mayo", 6: "Junio", 7: "Julio", 8: "Agosto",
    9: "Septiembre", 10: "Octubre", 11: "Noviembre", 12: "Diciembre",
}

MESES_CORTOS = {
    1: "Ene", 2: "Feb", 3: "Mar", 4: "Abr",
    5: "May", 6: "Jun", 7: "Jul", 8: "Ago",
    9: "Sep", 10: "Oct", 11: "Nov", 12: "Dic",
}

df["mes_nombre"] = df["mes"].map(MESES).astype("string")
df["mes_nombre_corto"] = df["mes"].map(MESES_CORTOS).astype("string")
df["trimestre"] = "T" + (((df["mes"].astype(int) - 1) // 3) + 1).astype(str)
df["anio_mes"] = df["fecha"].dt.strftime("%Y-%m").astype("string")
df["orden_anio_mes"] = (
    df["anio"].astype(int) * 100 + df["mes"].astype(int)
).astype("int32")

# Un año se considera completo cuando el dataset contiene sus 12 meses.
meses_por_anio = df.groupby("anio", observed=True)["mes"].nunique()
mapa_anio_completo = meses_por_anio.eq(12)

df["anio_completo"] = (
    df["anio"].map(mapa_anio_completo).fillna(False).astype(bool)
)
df["periodo_comparacion_2023_2025"] = df["anio"].between(2023, 2025)
df["periodo_principal_2025"] = df["anio"].eq(2025)

display(
    df[
        [
            "periodo", "fecha", "anio", "mes", "mes_nombre",
            "trimestre", "anio_completo",
        ]
    ].drop_duplicates().sort_values("fecha").tail(15)
)

,periodo,fecha,anio,mes,mes_nombre,trimestre,anio_completo
314,2025M03,2025-03-01,2025,3,Marzo,T1,True
315,2025M04,2025-04-01,2025,4,Abril,T2,True
316,2025M05,2025-05-01,2025,5,Mayo,T2,True
317,2025M06,2025-06-01,2025,6,Junio,T2,True
318,2025M07,2025-07-01,2025,7,Julio,T3,True
319,2025M08,2025-08-01,2025,8,Agosto,T3,True
320,2025M09,2025-09-01,2025,9,Septiembre,T3,True
321,2025M10,2025-10-01,2025,10,Octubre,T4,True
322,2025M11,2025-11-01,2025,11,Noviembre,T4,True
323,2025M12,2025-12-01,2025,12,Diciembre,T4,True


## 5. Correspondencia StaySpain ↔ unidad geográfica EOH

La correspondencia observada en los EDA queda incorporada como columnas del mismo clean dataset:

- Barcelona, Madrid, Valencia, Málaga y Sevilla → tabla 2078, punto turístico.
- Mallorca y Menorca → tabla 2039, zona insular.
- Girona → tabla 2039, `Costa Brava`, porque la oferta StaySpain representa la provincia/costa y no solo el municipio.

Las filas provinciales y autonómicas conservan los mercados relacionados sin duplicar registros.

In [15]:
clave_mercado = df["tabla"].astype(str) + "|" + df["geo"].astype(str)

MAPEO_MERCADO = {
    "2078|Barcelona": "Barcelona",
    "2078|Madrid": "Madrid",
    "2078|València": "Valencia",
    "2078|Málaga": "Málaga",
    "2078|Sevilla": "Sevilla",
    "2039|Isla De Mallorca": "Mallorca",
    "2039|Isla De Menorca": "Menorca",
    "2039|Costa Brava": "Girona",
}

MAPEO_TIPO_MERCADO = {
    clave: (
        "Ciudad"
        if clave.startswith("2078")
        else "Zona costera"
        if mercado == "Girona"
        else "Isla"
    )
    for clave, mercado in MAPEO_MERCADO.items()
}

PROVINCIA_POR_MERCADO = {
    "Barcelona": "Barcelona",
    "Madrid": "Madrid",
    "Valencia": "Valencia/València",
    "Málaga": "Málaga",
    "Sevilla": "Sevilla",
    "Girona": "Girona",
    "Mallorca": "Balears, Illes",
    "Menorca": "Balears, Illes",
}

CCAA_POR_MERCADO = {
    "Barcelona": "Cataluña",
    "Madrid": "Madrid, Comunidad de",
    "Valencia": "Comunitat Valenciana",
    "Málaga": "Andalucía",
    "Sevilla": "Andalucía",
    "Girona": "Cataluña",
    "Mallorca": "Balears, Illes",
    "Menorca": "Balears, Illes",
}

df["mercado_stayspain"] = clave_mercado.map(MAPEO_MERCADO).astype("string")
df["tipo_mercado"] = clave_mercado.map(MAPEO_TIPO_MERCADO).astype("string")
df["provincia_referencia"] = (
    df["mercado_stayspain"].map(PROVINCIA_POR_MERCADO).astype("string")
)
df["ccaa_referencia"] = (
    df["mercado_stayspain"].map(CCAA_POR_MERCADO).astype("string")
)

MAPEO_MERCADOS_RELACIONADOS = {
    "provincia|Barcelona": "Barcelona",
    "provincia|Madrid": "Madrid",
    "provincia|Girona": "Girona",
    "provincia|Valencia/València": "Valencia",
    "provincia|Málaga": "Málaga",
    "provincia|Sevilla": "Sevilla",
    "provincia|Balears, Illes": "Mallorca | Menorca",
    "ccaa|Cataluña": "Barcelona | Girona",
    "ccaa|Madrid, Comunidad de": "Madrid",
    "ccaa|Comunitat Valenciana": "Valencia",
    "ccaa|Andalucía": "Málaga | Sevilla",
    "ccaa|Balears, Illes": "Mallorca | Menorca",
}

clave_relacion = df["nivel_geo"].astype(str) + "|" + df["geo"].astype(str)
df["mercados_relacionados"] = (
    clave_relacion.map(MAPEO_MERCADOS_RELACIONADOS).astype("string")
)

PROVINCIA_A_CCAA = {
    "Barcelona": "Cataluña",
    "Madrid": "Madrid, Comunidad de",
    "Girona": "Cataluña",
    "Valencia/València": "Comunitat Valenciana",
    "Málaga": "Andalucía",
    "Sevilla": "Andalucía",
    "Balears, Illes": "Balears, Illes",
}

mascara_provincia = (
    df["nivel_geo"].eq("provincia")
    & df["geo"].isin(PROVINCIA_A_CCAA)
)
df.loc[mascara_provincia, "provincia_referencia"] = (
    df.loc[mascara_provincia, "geo"].astype("string")
)
df.loc[mascara_provincia, "ccaa_referencia"] = (
    df.loc[mascara_provincia, "geo"].map(PROVINCIA_A_CCAA).astype("string")
)

ccaa_interes = set(PROVINCIA_A_CCAA.values())
mascara_ccaa = df["nivel_geo"].eq("ccaa") & df["geo"].isin(ccaa_interes)
df.loc[mascara_ccaa, "ccaa_referencia"] = (
    df.loc[mascara_ccaa, "geo"].astype("string")
)

df["ambito_sprint4"] = (
    df["mercado_stayspain"].notna()
    | df["mercados_relacionados"].notna()
)
df["es_demanda_destino"] = df["mercado_stayspain"].notna()
df["es_procedencia_nacional"] = (
    df["tabla"].eq("2069")
    & df["nivel_geo"].eq("provincia")
    & df["mercados_relacionados"].notna()
)
df["es_total_territorial"] = (
    df["tabla"].eq("2074")
    & df["nivel_geo"].isin(["provincia", "ccaa"])
    & df["mercados_relacionados"].notna()
)

display(
    df.loc[
        df["es_demanda_destino"],
        [
            "tabla", "geo", "mercado_stayspain", "tipo_mercado",
            "provincia_referencia", "ccaa_referencia",
        ],
    ].drop_duplicates().sort_values("mercado_stayspain")
)

,tabla,geo,mercado_stayspain,tipo_mercado,provincia_referencia,ccaa_referencia
915276,2078,Barcelona,Barcelona,Ciudad,Barcelona,Cataluña
14476,2039,Costa Brava,Girona,Zona costera,Girona,Cataluña
968732,2078,Madrid,Madrid,Ciudad,Madrid,"Madrid, Comunidad de"
39480,2039,Isla De Mallorca,Mallorca,Isla,"Balears, Illes","Balears, Illes"
40796,2039,Isla De Menorca,Menorca,Isla,"Balears, Illes","Balears, Illes"
975928,2078,Málaga,Málaga,Ciudad,Málaga,Andalucía
1012936,2078,Sevilla,Sevilla,Ciudad,Sevilla,Andalucía
1029384,2078,València,Valencia,Ciudad,Valencia/València,Comunitat Valenciana


## 6. Estancia media y peso nacional/extranjero

La estancia media se calcula correctamente como:

```text
pernoctaciones totales / viajeros totales
```

No se utiliza la media simple de cocientes mensuales.

Los porcentajes nacional y extranjero se calculan dentro de cada geografía, periodo e indicador.

In [16]:
CLAVES_ESTANCIA = [
    "tabla", "nivel_geo", "geo_key", "categoria_key",
    "nivel_jerarquico", "periodo",
]

mascara_valores_absolutos = (
    df["tabla"].isin(["2039", "2074", "2078"])
    & df["metrica"].isin(["viajeros", "pernoctaciones"])
)

tabla_estancia = (
    df.loc[
        mascara_valores_absolutos,
        CLAVES_ESTANCIA + ["metrica", "valor"],
    ]
    .pivot_table(
        index=CLAVES_ESTANCIA,
        columns="metrica",
        values="valor",
        aggfunc="first",
    )
)

tabla_estancia["estancia_media"] = (
    tabla_estancia["pernoctaciones"]
    / tabla_estancia["viajeros"].replace(0, np.nan)
)

indice_estancia = pd.MultiIndex.from_frame(df[CLAVES_ESTANCIA])
df["estancia_media"] = (
    indice_estancia.map(tabla_estancia["estancia_media"]).astype("float64")
)


CLAVES_PORCENTAJE = [
    "tabla", "nivel_geo", "geo_key",
    "nivel_jerarquico", "periodo", "metrica",
]

mascara_residencia = (
    df["tabla"].isin(["2039", "2074", "2078"])
    & df["nivel_jerarquico"].eq("desglose")
    & df["categoria_key"].isin(
        ["residentes en espana", "residentes en el extranjero"]
    )
)

tabla_porcentajes = (
    df.loc[
        mascara_residencia,
        CLAVES_PORCENTAJE + ["categoria_key", "valor"],
    ]
    .pivot_table(
        index=CLAVES_PORCENTAJE,
        columns="categoria_key",
        values="valor",
        aggfunc="first",
    )
)

denominador = (
    tabla_porcentajes["residentes en espana"]
    + tabla_porcentajes["residentes en el extranjero"]
)

tabla_porcentajes["pct_nacional"] = (
    tabla_porcentajes["residentes en espana"]
    / denominador.replace(0, np.nan)
    * 100
)

tabla_porcentajes["pct_extranjero"] = (
    tabla_porcentajes["residentes en el extranjero"]
    / denominador.replace(0, np.nan)
    * 100
)

indice_porcentajes = pd.MultiIndex.from_frame(df[CLAVES_PORCENTAJE])
df["pct_nacional"] = (
    indice_porcentajes.map(tabla_porcentajes["pct_nacional"]).astype("float64")
)
df["pct_extranjero"] = (
    indice_porcentajes.map(tabla_porcentajes["pct_extranjero"]).astype("float64")
)

print("Filas con estancia media:", f"{df['estancia_media'].notna().sum():,}")
print("Filas con porcentajes de residencia:", f"{df['pct_nacional'].notna().sum():,}")

Filas con estancia media: 283,248
Filas con porcentajes de residencia: 237,350


## 7. Ponderación de la procedencia nacional: tablas 2069 + 2074

La tabla 2069 contiene porcentajes. Para evitar promediar meses con distinto volumen, esos porcentajes se aplican a los totales de residentes en España de la tabla 2074.

Los resultados se incorporan únicamente en las filas de la tabla 2069:

- `total_nacional_provincia`
- `valor_estimado_origen`
- `estancia_media_origen`

No se crean filas nuevas ni un CSV auxiliar.

In [17]:
CLAVES_TOTAL_PROVINCIA = ["geo_key", "periodo", "metrica"]

mascara_total_nacional = (
    df["tabla"].eq("2074")
    & df["nivel_geo"].eq("provincia")
    & df["nivel_jerarquico"].eq("desglose")
    & df["categoria_key"].eq("residentes en espana")
)

total_nacional_provincia = (
    df.loc[mascara_total_nacional]
    .set_index(CLAVES_TOTAL_PROVINCIA)["valor"]
)

mascara_2069_detalle = (
    df["tabla"].eq("2069")
    & df["nivel_geo"].eq("provincia")
    & df["nivel_jerarquico"].eq("desglose")
)

df["total_nacional_provincia"] = np.nan

indice_2069_total = pd.MultiIndex.from_frame(
    df.loc[mascara_2069_detalle, CLAVES_TOTAL_PROVINCIA]
)

df.loc[
    mascara_2069_detalle,
    "total_nacional_provincia",
] = indice_2069_total.map(total_nacional_provincia).to_numpy()

df["valor_estimado_origen"] = np.nan
df.loc[mascara_2069_detalle, "valor_estimado_origen"] = (
    df.loc[mascara_2069_detalle, "valor"]
    / 100
    * df.loc[mascara_2069_detalle, "total_nacional_provincia"]
)

CLAVES_ORIGEN = ["geo_key", "categoria_key", "periodo"]

tabla_estancia_origen = (
    df.loc[
        mascara_2069_detalle,
        CLAVES_ORIGEN + ["metrica", "valor_estimado_origen"],
    ]
    .pivot_table(
        index=CLAVES_ORIGEN,
        columns="metrica",
        values="valor_estimado_origen",
        aggfunc="first",
    )
)

tabla_estancia_origen["estancia_media_origen"] = (
    tabla_estancia_origen["pernoctaciones"]
    / tabla_estancia_origen["viajeros"].replace(0, np.nan)
)

df["estancia_media_origen"] = np.nan

indice_estancia_origen = pd.MultiIndex.from_frame(
    df.loc[mascara_2069_detalle, CLAVES_ORIGEN]
)

df.loc[
    mascara_2069_detalle,
    "estancia_media_origen",
] = indice_estancia_origen.map(
    tabla_estancia_origen["estancia_media_origen"]
).to_numpy()

print(
    "Filas 2069 con volumen estimado:",
    f"{df['valor_estimado_origen'].notna().sum():,}",
)

Filas 2069 con volumen estimado: 588,248


## 8. Temporada de demanda por mercado y procedencia

La recomendación del EDA es no asignar una temporada fija únicamente por mes calendario.

La clasificación se calcula para cada mercado y para cada procedencia —nacional o extranjera— usando la media mensual de pernoctaciones de 2023–2025:

- tercio inferior → `Baja`
- tercio intermedio → `Media`
- tercio superior → `Alta`

También se incorporan el ranking mensual, el índice estacional base 100 y el peso mensual.

In [18]:
CLAVES_TEMPORADA = ["tabla", "geo_key", "categoria_key", "mes"]

mascara_base_temporada = (
    df["es_demanda_destino"]
    & df["metrica"].eq("pernoctaciones")
    & df["nivel_jerarquico"].eq("desglose")
    & df["categoria_key"].isin(
        ["residentes en espana", "residentes en el extranjero"]
    )
    & df["anio"].between(2023, 2025)
    & df["valor"].notna()
)

tabla_temporada = (
    df.loc[
        mascara_base_temporada,
        CLAVES_TEMPORADA + ["valor"],
    ]
    .groupby(CLAVES_TEMPORADA, observed=True, as_index=False)["valor"]
    .mean()
)

CLAVES_MERCADO_ORIGEN = ["tabla", "geo_key", "categoria_key"]

tabla_temporada["ranking_mes_demanda_2023_2025"] = (
    tabla_temporada
    .groupby(CLAVES_MERCADO_ORIGEN, observed=True)["valor"]
    .rank(method="dense", ascending=False)
    .astype("int8")
)

media_grupo = (
    tabla_temporada
    .groupby(CLAVES_MERCADO_ORIGEN, observed=True)["valor"]
    .transform("mean")
)

suma_grupo = (
    tabla_temporada
    .groupby(CLAVES_MERCADO_ORIGEN, observed=True)["valor"]
    .transform("sum")
)

tabla_temporada["indice_estacional_base_100"] = (
    tabla_temporada["valor"] / media_grupo * 100
)

tabla_temporada["pct_mes_demanda_2023_2025"] = (
    tabla_temporada["valor"] / suma_grupo * 100
)

rango_pct = (
    tabla_temporada
    .groupby(CLAVES_MERCADO_ORIGEN, observed=True)["valor"]
    .rank(method="average", pct=True)
)

tabla_temporada["temporada_demanda"] = np.select(
    [
        rango_pct <= 1 / 3,
        rango_pct <= 2 / 3,
    ],
    [
        "Baja",
        "Media",
    ],
    default="Alta",
)

tabla_temporada = tabla_temporada.set_index(CLAVES_TEMPORADA)
indice_temporada = pd.MultiIndex.from_frame(df[CLAVES_TEMPORADA])

for columna in [
    "temporada_demanda",
    "ranking_mes_demanda_2023_2025",
    "indice_estacional_base_100",
    "pct_mes_demanda_2023_2025",
]:
    df[columna] = indice_temporada.map(tabla_temporada[columna])

display(
    df.loc[
        df["es_demanda_destino"] & df["temporada_demanda"].notna(),
        [
            "mercado_stayspain", "categoria", "mes", "mes_nombre",
            "temporada_demanda", "ranking_mes_demanda_2023_2025",
            "indice_estacional_base_100",
        ],
    ]
    .drop_duplicates()
    .sort_values(
        ["mercado_stayspain", "categoria", "ranking_mes_demanda_2023_2025"]
    )
    .head(24)
)

,mercado_stayspain,categoria,mes,mes_nombre,temporada_demanda,ranking_mes_demanda_2023_2025,indice_estacional_base_100
915278,Barcelona,Residentes en España,3,Marzo,Alta,1.0,111.959679
915287,Barcelona,Residentes en España,12,Diciembre,Alta,2.0,111.074416
915282,Barcelona,Residentes en España,7,Julio,Alta,3.0,108.850014
915286,Barcelona,Residentes en España,11,Noviembre,Alta,4.0,107.878049
915277,Barcelona,Residentes en España,2,Febrero,Media,5.0,103.174232
915276,Barcelona,Residentes en España,1,Enero,Media,6.0,102.974662
915285,Barcelona,Residentes en España,10,Octubre,Media,7.0,101.142469
915280,Barcelona,Residentes en España,5,Mayo,Media,8.0,97.998010
915279,Barcelona,Residentes en España,4,Abril,Baja,9.0,95.261318
915281,Barcelona,Residentes en España,6,Junio,Baja,10.0,91.054374


## 9. Validaciones finales

Antes de reemplazar el clean dataset se comprueba que:

- no se ha perdido ni duplicado ninguna fila;
- todos los periodos tienen fecha válida;
- los nulos numéricos están explicados por `sin_dato`;
- los ocho mercados StaySpain están representados;
- los porcentajes nacional y extranjero suman aproximadamente 100;
- los porcentajes de procedencia de la tabla 2069 son coherentes.

In [19]:
CLAVE_UNICA = [
    "tabla", "nivel_geo", "geo_key", "dimension",
    "categoria_key", "metrica", "nivel_jerarquico", "periodo",
]

filas_duplicadas = int(df.duplicated(CLAVE_UNICA).sum())
fechas_invalidas = int(df["fecha"].isna().sum())
nulos_no_explicados = int(
    (df["valor"].isna() & ~df["sin_dato"]).sum()
)

mercados_detectados = sorted(
    df["mercado_stayspain"].dropna().unique().tolist()
)

desviacion_residencia = (
    (
        tabla_porcentajes["pct_nacional"]
        + tabla_porcentajes["pct_extranjero"]
        - 100
    )
    .abs()
    .max()
)

control_2069 = (
    df.loc[
        df["tabla"].eq("2069")
        & df["nivel_geo"].eq("provincia")
        & df["nivel_jerarquico"].eq("desglose")
        & df["anio"].between(2023, 2025)
        & df["valor"].notna()
    ]
    .groupby(["geo_key", "periodo", "metrica"], observed=True)["valor"]
    .sum(min_count=1)
)

# Se excluyen grupos totalmente nulos o sin actividad publicada.
control_2069_publicado = control_2069[control_2069 > 0]

grupos_2069_fuera_rango = int(
    (
        (control_2069_publicado < 99)
        | (control_2069_publicado > 101)
    ).sum()
)

assert len(df) == filas_iniciales, "Se ha modificado el número de filas."
assert filas_duplicadas == 0, "Se han generado duplicados."
assert fechas_invalidas == 0, "Hay fechas inválidas."
assert nulos_no_explicados == 0, "Hay nulos numéricos no explicados."
assert len(mercados_detectados) == 8, "No se detectaron los 8 mercados."
assert desviacion_residencia <= 0.01, "Los porcentajes de residencia no suman 100."
assert grupos_2069_fuera_rango == 0, "Hay grupos 2069 fuera del rango 99–101."

validaciones = pd.DataFrame(
    {
        "control": [
            "Filas conservadas",
            "Duplicados",
            "Fechas inválidas",
            "Nulos no explicados",
            "Mercados StaySpain",
            "Desviación máxima nacional + extranjero",
            "Grupos 2069 fuera de 99–101",
        ],
        "resultado": [
            len(df),
            filas_duplicadas,
            fechas_invalidas,
            nulos_no_explicados,
            len(mercados_detectados),
            round(float(desviacion_residencia), 8),
            grupos_2069_fuera_rango,
        ],
        "estado": ["OK"] * 7,
    }
)

display(validaciones)
print("Mercados detectados:", ", ".join(mercados_detectados))

,control,resultado,estado
0,Filas conservadas,1040692.0,OK
1,Duplicados,0.0,OK
2,Fechas inválidas,0.0,OK
3,Nulos no explicados,0.0,OK
4,Mercados StaySpain,8.0,OK
5,Desviación máxima nacional + extranjero,0.0,OK
6,Grupos 2069 fuera de 99–101,0.0,OK


Mercados detectados: Barcelona, Girona, Madrid, Mallorca, Menorca, Málaga, Sevilla, Valencia


## 10. Reescritura del único clean dataset final

El fichero se guarda con **el mismo nombre y en la misma ubicación** que el clean dataset de entrada:

```text
Data/clean_dataset_INE_EOH_20_07_2026.csv
```

La escritura es atómica: primero se crea un temporal y, cuando la exportación termina correctamente, reemplaza al CSV anterior. El temporal no permanece en la carpeta.

No se genera ningún otro CSV.

In [20]:
COLUMNAS_FINALES = [
    "tabla", "nivel_geo", "codigo_ine", "geo", "geo_key", "zona_ccaa",
    "dimension", "categoria", "categoria_key", "metrica", "unidad",
    "nivel_jerarquico", "periodo", "fecha", "anio", "mes",
    "mes_nombre", "mes_nombre_corto", "trimestre", "anio_mes",
    "orden_anio_mes", "anio_completo",
    "periodo_comparacion_2023_2025", "periodo_principal_2025",
    "valor", "sin_dato", "es_dato_no_publicado",
    "provisional", "serie_enlazada",
    "mercado_stayspain", "tipo_mercado",
    "provincia_referencia", "ccaa_referencia",
    "mercados_relacionados", "ambito_sprint4",
    "es_demanda_destino", "es_procedencia_nacional", "es_total_territorial",
    "estancia_media", "pct_nacional", "pct_extranjero",
    "total_nacional_provincia", "valor_estimado_origen",
    "estancia_media_origen", "temporada_demanda",
    "ranking_mes_demanda_2023_2025",
    "indice_estacional_base_100", "pct_mes_demanda_2023_2025",
]

columnas_finales_ausentes = [
    columna for columna in COLUMNAS_FINALES if columna not in df.columns
]
if columnas_finales_ausentes:
    raise ValueError(
        f"No se pudieron crear estas columnas: {columnas_finales_ausentes}"
    )

salida = (
    df[COLUMNAS_FINALES]
    .sort_values(
        [
            "tabla", "nivel_geo", "geo",
            "dimension", "categoria", "metrica", "fecha",
        ]
    )
    .reset_index(drop=True)
)

ruta_temporal = RUTA_DATASET.with_name(
    f"{RUTA_DATASET.stem}__temporal.csv"
)

salida.to_csv(
    ruta_temporal,
    index=False,
    sep=";",
    decimal=",",
    encoding="utf-8-sig",
    date_format="%Y-%m-%d",
)

os.replace(ruta_temporal, RUTA_DATASET)

print(f"Dataset final actualizado: {NOMBRE_DATASET}")
print(f"Filas: {len(salida):,}")
print(f"Columnas: {len(salida.columns)}")

Dataset final actualizado: clean_dataset_INE_EOH_20_07_2026.csv
Filas: 1,040,692
Columnas: 48


## Resultado

El proceso termina con un único fichero final enriquecido:

```text
clean_dataset_INE_EOH_20_07_2026.csv
```

El dataset:

- conserva toda la información limpiada;
- añade las variables temporales nominales;
- integra las transformaciones requeridas por los EDA;
- mantiene explícita la granularidad de las cuatro tablas;
- permite filtrar los ocho mercados de StaySpain;
- prepara los cálculos de procedencia, estacionalidad y estancia media;
- y reemplaza al clean dataset anterior sin crear outputs auxiliares.